# AIC25 — Tier-2: 3D-HOTA Numbers

Produces the official **3D-HOTA** score for Warehouse_016 (paper-comparable; Glance-MCMT scored 43–51).

**Same fixes as Tier-1** (OSNet from HF mirror, EmbedFeature local, fresh capped detection) **plus** depth maps + multi-camera + TrackEval.

⚠️ **Heavy path:**
1. **Depth maps** (30–80 GB) — required for 3D world coordinates.
2. **All cameras** — multi-camera HOTA needs every view (`CAMERAS = None`).
3. Single-camera tracking re-run **with depth present**.

`MAXF` caps frames for a faster (partial) HOTA; set `MAXF = 0` for full 9000-frame, paper-faithful HOTA (very long on free Colab).

Branch: **`hithesh/combined-pipeline`**. T4 GPU.

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys, shutil
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_COLAB:
    REPO='/content/repo'; PY='python'; DRIVE='/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
    else: print('Drive already mounted.')
    for d in ['models','outputs/Detection','outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print('Colab | Drive:', DRIVE)
else:
    REPO='/home/seco/deepLearning/Single-Camera-Tracking-Consistency'; PY=f'{REPO}/.venv/bin/python'; DRIVE=None
    os.chdir(REPO); print('Local')
print('REPO:', REPO)

---
## Step 1 — Clone + checkout combined branch + install

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
    else:
        os.system(f'git -C {REPO} fetch --quiet')
    os.chdir(REPO)
    rc = os.system(f'git -C {REPO} checkout hithesh/combined-pipeline')
    os.system(f'git -C {REPO} pull --quiet 2>/dev/null')
    if rc != 0 or not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError('Checkout failed — push: git push -u origin hithesh/combined-pipeline')
    print('On hithesh/combined-pipeline ✓')
    for _p in ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs','termcolor',
               'prettytable','tabulate','ninja','cython_bbox','pycocotools','huggingface_hub','h5py']:
        _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
    if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
        _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
    os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(REPO); os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
    print('Dependencies installed.')
else:
    print('Local: skip.')

---
## Step 2 — GPU check

In [ ]:
import subprocess
r = subprocess.run([PY,'-c','import torch; print("CUDA:", torch.cuda.is_available())'], capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — switch to T4, Restart, re-run.')

---
## Step 3 — Models (OSNet from HF mirror + ByteTrack; AIC25 detector if trained)

In [ ]:
if ON_COLAB:
    from huggingface_hub import hf_hub_download
    M=f'{DRIVE}/models'; os.makedirs(M, exist_ok=True)
    osnet_local=f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'; osnet_drive=f'{M}/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet_local), exist_ok=True)
    if os.path.exists(osnet_local): print('OSNet: local')
    elif os.path.exists(osnet_drive): shutil.copy(osnet_drive, osnet_local); print('OSNet: from Drive')
    else:
        fn='osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth'
        src=hf_hub_download(repo_id='kaiyangzhou/osnet', filename=fn)
        shutil.copy(src, osnet_local); shutil.copy(osnet_local, osnet_drive); print('OSNet: from HF mirror')
    bt_local=f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; bt_drive=f'{M}/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bt_local), exist_ok=True)
    if not os.path.exists(bt_local):
        if os.path.exists(bt_drive): shutil.copy(bt_drive, bt_local)
        else:
            os.system('pip install -q -U gdown'); import gdown
            gdown.download(id='1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', output=bt_local, quiet=False)
            if os.path.exists(bt_local): shutil.copy(bt_local, bt_drive)
    aic=f'{M}/ai_city_ckpt.pth.tar'
    if os.path.exists(aic): shutil.copy(aic, f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'); print('AIC25 detector: from Drive ✓')
    else: print('AIC25 detector: not trained — ByteTrack fallback (HOTA will be lower)')
else: print('Local: models in place.')

---
## Step 4 — Config (all cameras for multi-cam HOTA)

In [ ]:
SCENE='Warehouse_016'; DATASET='Val'
CAMERAS=None          # None = all cameras (REQUIRED for multi-camera HOTA)
MAXF=1500             # frames/camera; 0 = full 9000 (paper-faithful but very long)
os.chdir(REPO)
print(f'{SCENE} ({DATASET}) | cameras ALL | cap {MAXF or "ALL 9000"} frames')

---
## Step 5 — Download videos + ground_truth (HuggingFace)

In [ ]:
if ON_COLAB:
    import getpass
    from huggingface_hub import snapshot_download, login
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    dv=f'{dd}/videos'; vl=f'{ld}/videos'
    os.makedirs(dd, exist_ok=True); os.makedirs(ld, exist_ok=True)
    if os.path.exists(dv) and os.listdir(dv): print('[CACHE HIT] videos on Drive.')
    else:
        tok=None
        try:
            from google.colab import userdata; tok=userdata.get('HF_TOKEN')
        except Exception: pass
        if not tok: tok=getpass.getpass('HF token: ')
        login(token=tok); sp=DATASET.lower()
        print('Downloading videos + GT...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_tmp',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/videos/**',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/calibration.json',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/ground_truth.json'])
        src=f'/content/hf_tmp/MTMC_Tracking_2025/{sp}/{SCENE}'
        if not os.path.exists(dv): shutil.copytree(f'{src}/videos', dv)
        for fn in ['calibration.json','ground_truth.json']:
            if os.path.exists(f'{src}/{fn}'): shutil.copy(f'{src}/{fn}', f'{dd}/{fn}')
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    for fn in ['calibration.json','ground_truth.json']:
        s,d=f'{dd}/{fn}',f'{ld}/{fn}'
        if os.path.exists(s) and not os.path.exists(d): shutil.copy(s,d)
    if os.path.islink(vl): os.unlink(vl)
    os.makedirs(vl, exist_ok=True)
    cams=sorted(os.path.splitext(f)[0] for f in os.listdir(dv) if f.endswith('.mp4'))
    print(f'✓ {len(cams)} cameras: {cams}')
else: print('Local: existing data.')

---
## Step 6 — Download DEPTH MAPS ⚠️ (30–80 GB)
Required for 3D world coords. HF stores them as `depth_map*` ; code expects `depth_map/<Camera>.h5`. Cached to Drive.

In [ ]:
if ON_COLAB:
    import glob
    from huggingface_hub import snapshot_download
    sp=DATASET.lower()
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    drive_depth=f'{dd}/depth_map'; local_depth=f'{ld}/depth_map'; os.makedirs(drive_depth, exist_ok=True)
    if any(f.endswith('.h5') for f in os.listdir(drive_depth)):
        print('[CACHE HIT] depth on Drive:', len([f for f in os.listdir(drive_depth) if f.endswith('.h5')]), 'files')
    else:
        print('Downloading depth maps (30-80 GB, first time only)...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_depth',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/depth_map*/**'])
        found=glob.glob(f'/content/hf_depth/MTMC_Tracking_2025/{sp}/{SCENE}/**/*.h5', recursive=True)
        print(f'  found {len(found)} .h5')
        for p in found: shutil.copy(p, f'{drive_depth}/{os.path.basename(p)}')
        shutil.rmtree('/content/hf_depth', ignore_errors=True)
    if os.path.islink(local_depth): os.unlink(local_depth)
    elif os.path.isdir(local_depth): shutil.rmtree(local_depth, ignore_errors=True)
    os.symlink(drive_depth, local_depth)
    h5=[f for f in os.listdir(local_depth) if f.endswith('.h5')]
    print(f'✓ depth_map/ ready — {len(h5)} files, e.g. {h5[:3]}')
    if not h5: print('⚠ no .h5 — check HF folder names; adjust allow_patterns.')
else: print('Local: depth_map/ expected under AIC25_Track1/...')

---
## Step 7 — Link outputs (Detection + Tracking → Drive; EmbedFeature → LOCAL)

In [ ]:
if ON_COLAB:
    for folder in ['Detection','Tracking']:
        df=f'{DRIVE}/outputs/{folder}'; rf=f'{REPO}/{folder}'; os.makedirs(df, exist_ok=True)
        if os.path.islink(rf): pass
        elif os.path.isdir(rf):
            for it in os.listdir(rf):
                s,d=f'{rf}/{it}',f'{df}/{it}'
                if not os.path.exists(d): shutil.move(s,d)
            shutil.rmtree(rf, ignore_errors=True); os.symlink(df, rf)
        else: os.symlink(df, rf)
        print(f'{folder}/ → Drive')
    ef=f'{REPO}/EmbedFeature'
    if os.path.islink(ef): os.unlink(ef)
    os.makedirs(ef, exist_ok=True); print('EmbedFeature/ → LOCAL')
else: print('Local.')

---
## Generate single-camera JSONs **with depth** (all cameras, capped)
Depth present → tracking computes 3D world coordinates (needed for HOTA). Fresh capped detection avoids stale-cache hangs.

In [ ]:
import subprocess
os.chdir(REPO)
_dv=f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos' if DRIVE else None
_lv=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
cam_source=_dv if (_dv and os.path.exists(_dv)) else _lv
all_cams=sorted(os.path.splitext(f)[0] for f in os.listdir(cam_source) if f.endswith('.mp4'))
cams=[c for c in CAMERAS if c in all_cams] if CAMERAS else all_cams
print('Cameras:', cams, '| cap', MAXF or 'ALL')
if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt='BoT-SORT/ai_city_ckpt.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
else:
    ckpt='BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'
det_cap=f'--max_frames {MAXF}' if MAXF else ''; trk_cap=f'--limit_frames {MAXF}' if MAXF else ''
shutil.rmtree(f'{REPO}/Detection/{SCENE}', ignore_errors=True);  os.makedirs(f'{REPO}/Detection/{SCENE}', exist_ok=True)
shutil.rmtree(f'{REPO}/EmbedFeature/{SCENE}', ignore_errors=True)
def run(cmd):
    r=subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode!=0: print('\n'.join(r.stdout.strip().splitlines()[-30:]))
    return r.returncode
for cam in cams:
    print(f'\n=== detect {cam} ===')
    run(f'{PY} BoT-SORT/tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} --camera {cam} -f {exp} -c {ckpt} {det_cap} ./')
print('\n=== [D] embeddings ===')
os.chdir(f'{REPO}/deep-person-reid'); print('rc', os.system(f'{PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')); os.chdir(REPO)
for cam in cams:
    print(f'\n=== track+fix {cam} ===')
    run(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET} {trk_cap}')
    run(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
print('\n[DONE] single-camera (with world coords) ready.')

---
## [G] Multi-camera tracking + fix
`multi_camera_revised.py` auto-creates `expN` — we detect it and pass it to `multi_camera_fix.py --exp_path`. `--total_frames` is set to the cap so paths line up.

In [ ]:
import os
os.chdir(REPO)
TF = MAXF if MAXF else 9000
mc_dir=f'{REPO}/Tracking/Multicamera/{SCENE}'
before=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
print('[G] multi_camera_revised...')
print('  rc', os.system(f'{PY} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET} --total_frames {TF}'))
after=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
EXP=sorted(after-before)[-1] if (after-before) else (sorted(after)[-1] if after else 'exp1')
print('  exp dir =', EXP)
print('[G] multi_camera_fix...')
print('  rc', os.system(f'{PY} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET} --exp_path {EXP}'))
ed=f'{mc_dir}/{EXP}'
print('  exp dir files:', os.listdir(ed) if os.path.isdir(ed) else 'MISSING')
orr=f'{ed}/output_result'
if os.path.isdir(orr): print('  output_result/:', os.listdir(orr))

---
## [H] Evaluate — 3D HOTA
Uses the **same EXP** from [G]. If `fixed_whole_tracking_results.json` isn't found, the [G] file listing shows what `multi_camera_fix` actually produced — adjust if needed.

In [ ]:
import os
os.chdir(f'{REPO}/TrackEval')
print('[H] prepare_eval_data (EXP=%s)...' % EXP)
print('  rc', os.system(f'{PY} prepare_eval_data.py -s {SCENE} --exp {EXP} --dataset {DATASET} --base_dir {REPO}'))
tt=f'{REPO}/TrackEval/aicity_25_data/{SCENE}/{EXP}.txt'
print('  tracker txt exists:', os.path.exists(tt))
print('[H] main (HOTA)...')
print('  rc', os.system(f'{PY} main.py -s {SCENE} --exp {EXP}'))
print('\n--- HOTA / DetA / AssA / LocA printed above ---')

---
## Notes
- **Before/after-repair HOTA** (showing `tracklet_repair` improves the score) needs the repaired single-camera JSON fed back into multi-camera — a schema-adaptation step, not automated here. Do the baseline HOTA first.
- For **paper-faithful** HOTA set `MAXF = 0` (full 9000 frames) — expect a multi-hour run; do it overnight or on a longer session.